# Command line

> The `itemiset` command: a file of sets in, a figure out.

In [ ]:
#| default_exp cli

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *
import tempfile

Installing the package adds an `itemiset` command:

```sh
itemiset hits.csv -o hits.png
itemiset hits.xlsx --sheet sets --bottom DDX6 -o figure.pdf
itemiset hits.csv --show-empty --title "P-body baits" -o figure.svg --zones-out zones.csv
```

The input can be any file `load_sets` reads. The output format follows the extension of `-o`.

In [ ]:
#| export
import argparse
import sys
from pathlib import Path

from itemiset import __version__
from itemiset.readers import load_sets
from itemiset.render import plot

In [ ]:
#| export
def _parser():
    ap = argparse.ArgumentParser(
        prog="itemiset",
        description="Draw an itemized area-proportional Euler diagram: one cell per item, so area is count.")
    ap.add_argument("input", help="CSV, TSV or Excel file: a membership table, or one column per set")
    ap.add_argument("-o", "--out", default=None, help="output .png, .pdf or .svg (default: <input>_itemiset.png)")
    ap.add_argument("--sheet", default=None, help="Excel sheet with the sets (default: the first sheet)")
    ap.add_argument("--sets", default=None,
                    help="comma-separated set columns to use, in colour order (cyan, magenta, yellow)")
    ap.add_argument("--categories", default=None, help="file with columns gene, category[, color]")
    ap.add_argument("--categories-sheet", default=None, help="sheet of the categories file to read")
    ap.add_argument("--category-colors", default=None, help='for example "Decapping=#C8102E,CCR4-NOT=#2140A8"')
    ap.add_argument("--bottom", default=None, help="set to place along the bottom (3 sets only)")
    ap.add_argument("--show-empty", action="store_true", help="draw each empty zone as one ∅ cell")
    ap.add_argument("--title", default=None, help="title above the diagram")
    ap.add_argument("--aspect", type=float, default=2.2, help="target width-to-height ratio (default 2.2)")
    ap.add_argument("--fontsize", type=float, default=6.5, help="font size of the names in points")
    ap.add_argument("--no-outline", action="store_true", help="don't draw set outlines")
    ap.add_argument("--zones-out", default=None, help="also write a CSV of each item and its zone")
    ap.add_argument("--version", action="version", version=f"itemiset {__version__}")
    return ap

In [ ]:
#| export
def run(argv=None):
    "Run the `itemiset` command with `argv` (default: the command-line arguments) and return the `Layout` it drew."
    ap = _parser()
    a = ap.parse_args(argv)
    sets, cats, ccols = load_sets(a.input, a.sheet, a.categories, a.categories_sheet)
    if a.sets:
        wanted = [s.strip() for s in a.sets.split(",")]
        missing = [s for s in wanted if s not in sets]
        if missing:
            ap.error(f"unknown set(s) {missing}; the file has {list(sets)}")
        sets = {s: sets[s] for s in wanted}
    if a.category_colors:
        for kv in a.category_colors.split(","):
            k, v = kv.rsplit("=", 1)
            ccols[k.strip()] = v.strip()
    out = a.out or str(Path(a.input).with_suffix("")) + "_itemiset.png"
    return plot(sets, out=out, categories=cats, category_colours=ccols, show_empty=a.show_empty,
                bottom=a.bottom, title=a.title, target_aspect=a.aspect, fontsize=a.fontsize,
                zones_out=a.zones_out, verbose=True, outline=not a.no_outline)

def main(argv=None):
    "Entry point for the `itemiset` command. Returns nothing, so the command exits with status 0."
    run(argv)

## Example

In [ ]:
tmp = Path(tempfile.mkdtemp())
(tmp/"hits.csv").write_text("A,B,C\nx1,x1,z1\nx2,y1,z2\nx3,y2,x1\n")
lay = run([str(tmp/"hits.csv"), "-o", str(tmp/"hits.png"), "--zones-out", str(tmp/"zones.csv"), "--bottom", "C"])
assert (tmp/"hits.png").exists()
test_eq(lay.roles["C"], "C")
print((tmp/"zones.csv").read_text())

Wrote /tmp/tmph0zpxjcn/hits.png
Roles: A=A, B=B, C=C | group widths (L, M, R): (1, 1, 1)
  A                                                 2
  B                                                 2
  C                                                 2
  A ∩ B ∩ C                                         1
item,zone,A,B,C,category
x2,A,1,0,0,
x3,A,1,0,0,
x1,A & B & C,1,1,1,
y1,B,0,1,0,
y2,B,0,1,0,
z1,C,0,0,1,
z2,C,0,0,1,



In [ ]:
# an unknown set name stops with a usage error (exit code 2)
try:
    run([str(tmp/"hits.csv"), "--sets", "A,Q"]); raise AssertionError("expected an error")
except SystemExit as e:
    test_eq(e.code, 2)

usage: itemiset [-h] [-o OUT] [--sheet SHEET] [--sets SETS]
                [--categories CATEGORIES]
                [--categories-sheet CATEGORIES_SHEET]
                [--category-colors CATEGORY_COLORS] [--bottom BOTTOM]
                [--show-empty] [--title TITLE] [--aspect ASPECT]
                [--fontsize FONTSIZE] [--no-outline] [--zones-out ZONES_OUT]
                [--version]
                input
itemiset: error: unknown set(s) ['Q']; the file has ['A', 'B', 'C']


`main` is what the installed `itemiset` command calls. It returns `None`, so the command exits with status 0; use `run` to get the `Layout` back.

In [ ]:
test_eq(main([str(tmp/"hits.csv"), "-o", str(tmp/"hits2.png")]), None)

Wrote /tmp/tmph0zpxjcn/hits2.png
Roles: A=B, B=C, C=A | group widths (L, M, R): (1, 1, 1)
  A                                                 2
  B                                                 2
  C                                                 2
  A ∩ B ∩ C                                         1


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()